# AMRS (Adaptive Model Recovery System) - Colab Bootstrap

**Security Boundary Notice**:
- FastAPI runs privately on `http://127.0.0.1:8000` (internal).
- Only the Next.js frontend is exposed via public tunnel on port `3000`.
- No private backend port is exposed directly to the internet.

In [ ]:
# 1. Verify GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# 2. Clone repository (if running in fresh Colab instance)
import os
if not os.path.exists("AMRS"):
    !git clone https://github.com/manthanrailkar/AMRS.git
    %cd AMRS
else:
    %cd AMRS
!pwd

In [ ]:
# 3. Install AMRS package and requirements
!pip install -e .
!amrs --version

In [ ]:
# 4. Build reproducible scenarios (S1-S6)
!amrs scenarios build --all --profile tiny --device cuda

In [ ]:
# 5. Launch FastAPI backend on internal private port 8000
import subprocess, time
backend_proc = subprocess.Popen(
    ["uvicorn", "amrs.api.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)
!curl -s http://127.0.0.1:8000/api/v1/health

In [ ]:
# 6. Launch Next.js web application on port 3000
%cd apps/web
!npm install
!npm run build
web_proc = subprocess.Popen(
    ["npm", "run", "start", "--", "-p", "3000"],
    env={**os.environ, "AMRS_API_INTERNAL_URL": "http://127.0.0.1:8000"}
)
time.sleep(5)
%cd ../..

In [ ]:
# 7. Expose EXCLUSIVELY Next.js (port 3000) through Cloudflared tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print("Starting tunnel on port 3000 (Next.js dashboard only)...")
!cloudflared tunnel --url http://127.0.0.1:3000